# PersonaPlex `<ref>` Context-Injection LoRA Training

Trains (or re-trains) the reference LoRA adapter that teaches PersonaPlex to
correctly consume `<ref>...</ref>` / `<lookup>...</lookup>` injected context,
using the dataset produced by `01_Dataset_Generation.ipynb`.

**This notebook downloads everything it needs itself** -- the PersonaPlex
4-bit weights and the bundled `moshi` source it depends on -- so a fresh
RunPod pod with nothing pre-installed can run it end to end. It downloads
only what training actually needs (the language model + tokenizer), skipping
the renderer/video/voice assets the full `prepare_imtalker_personaplex.sh`
also fetches for the *live avatar server*, which this notebook never runs.

**Multi-GPU:** Section 5 (training) auto-detects every GPU visible to the pod
and launches PyTorch `DistributedDataParallel` training across all of them via
`torchrun` -- each GPU loads its own full 4-bit copy of the model and works
on its own shard of the data, with gradients averaged across GPUs every
optimizer step. This is what actually drives every GPU to ~100% utilization
and scales training throughput roughly linearly with GPU count (2, 3, 4, 5+),
as opposed to `device_map="auto"`-style model sharding, which only helps a
model that doesn't fit on one GPU and does not speed up training.

**Before you run this on real training data, read Section 4 (the contract
check).** The model-loading code here is copied directly from your production
`IMTalker/liveTry.py`, so it is guaranteed to match. The training
forward/loss call targets the standard Moshi-family `LMModel` training
contract, but PersonaPlex is NVIDIA's own checkpoint/fork and could not be
verified against its exact source offline -- the contract check runs one real
forward+backward pass against your actual downloaded model on this pod and
tells you immediately whether that assumption holds, before any GPU time is
spent on real training. See the long comment at the top of
`ref_lora_training/common/model_adapter.py` for exactly what to change if it
does not.

**Output:** a LoRA adapter at `<REF_LORA_DIR>/lora/adapter_config.json` +
`adapter_model.safetensors` -- point your `run_imtalker_personaplex.sh`'s
`REF_LORA_DIR` at the parent folder to use it.

## 1. Environment setup

Installs only what's needed to load PersonaPlex and train a LoRA on it (not the full avatar/renderer stack).

In [ ]:
import subprocess, sys, re

def _pip(args):
    print("+ pip " + " ".join(args), flush=True)
    result = subprocess.run([sys.executable, "-m", "pip"] + args)
    if result.returncode != 0:
        raise RuntimeError(f"pip {args[0]} failed with exit code {result.returncode}")

_pip(["install", "-q", "--upgrade", "pip", "wheel"])
_pip(["install", "-q", "setuptools==80.9.0"])
# Deliberately NOT pinning numpy here. An earlier version of this cell forced
# numpy<2.0.0 (copying a constraint from IMTalker/requirement.txt, which this
# training notebook does not install) and that downgrade broke on a RunPod pod
# whose base image ships numpy 2.x with other pre-installed compiled packages
# already linked against numpy 2.x's ABI -- downgrading numpy afterward left
# those compiled extensions expecting a larger dtype struct than the now-older
# numpy actually provides ("ValueError: numpy.dtype size changed ... Expected 96
# ... got 88"), a classic numpy 1.x/2.x ABI mismatch. None of the packages this
# notebook actually needs (torch 2.8, bitsandbytes 0.50, transformers 4.52,
# peft, accelerate, moshi) require numpy<2, so just let pip resolve it.

# --- Pick a torch CUDA build that matches what THIS pod's driver actually
# supports, instead of blindly installing the cu128 build
# prepare_imtalker_personaplex.sh assumes for its known-good RTX 5090 pods.
# Installing a newer CUDA build than the driver supports doesn't fail at
# install time -- pip happily downloads it -- it fails later and confusingly,
# as `torch.cuda.is_available()` silently returning False with a buried
# "CUDA initialization: driver is too old" warning, which is exactly what
# produced DEVICE="cpu" here before this check existed.
driver_cuda = None
try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=15)
    m = re.search(r"CUDA Version:\s*([\d.]+)", smi.stdout)
    if m:
        driver_cuda = tuple(int(x) for x in m.group(1).split("."))
        print(f"nvidia-smi reports driver supports up to CUDA {m.group(1)}", flush=True)
    else:
        print("Could not parse a CUDA version out of nvidia-smi output.", flush=True)
except Exception as e:
    print(f"nvidia-smi check failed ({e!r}) -- assuming a recent driver.", flush=True)

if driver_cuda is None:
    torch_index = "https://download.pytorch.org/whl/cu128"
elif driver_cuda >= (12, 8):
    torch_index = "https://download.pytorch.org/whl/cu128"
elif driver_cuda >= (12, 4):
    torch_index = "https://download.pytorch.org/whl/cu124"
elif driver_cuda >= (12, 1):
    torch_index = "https://download.pytorch.org/whl/cu121"
else:
    torch_index = "https://download.pytorch.org/whl/cu118"
    print(
        f"WARNING: driver only supports CUDA {'.'.join(map(str, driver_cuda))}, which is quite "
        "old for a 2026 RunPod GPU pod. Training a 7B model needs a real GPU -- if the cell "
        "below still reports cuda available: False, stop and switch to a different RunPod "
        "GPU/template with a newer NVIDIA driver rather than continuing on CPU.",
        flush=True,
    )
print(f"Installing torch against index: {torch_index}", flush=True)

_pip(["install", "-q", "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0",
      "--index-url", torch_index])
_pip(["install", "-q", "huggingface_hub[cli]==0.36.2", "hf_transfer", "sphn==0.2.1",
      "einops", "sentencepiece", "bitsandbytes==0.50.0"])
_pip(["install", "-q", "peft>=0.19,<0.20", "transformers==4.52.4", "safetensors", "accelerate"])
print("Environment ready.")

In [ ]:
import numpy, torch, transformers, peft

print("numpy:", numpy.__version__)
cuda_ok = torch.cuda.is_available()
print("torch:", torch.__version__, " torch's CUDA build:", torch.version.cuda,
      " cuda available:", cuda_ok, " gpu count:", torch.cuda.device_count())
print("transformers:", transformers.__version__, " peft:", peft.__version__)

if not cuda_ok:
    raise RuntimeError(
        "torch.cuda.is_available() is False -- training a 7B model on CPU is not practical. "
        "This almost always means the torch CUDA build installed above is newer than what this "
        "pod's NVIDIA driver supports (see the 'nvidia-smi reports driver supports up to CUDA "
        "...' line printed in the cell above, and compare it against 'torch's CUDA build' just "
        "printed here). Fix by either: (a) re-running the previous cell after confirming the "
        "detected driver version looks right, or (b) terminating this pod and picking a RunPod "
        "GPU/template with a newer NVIDIA driver -- do not continue past this cell on CPU."
    )

## 2. Hugging Face access

`nvidia/personaplex-7b-v1` is a gated repo -- your Hugging Face account needs approved access to it, and a read token.

In [ ]:
import getpass, os

HF_TOKEN = os.getenv("HF_TOKEN", "_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (read access to nvidia/personaplex-7b-v1): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print("HF token set." if HF_TOKEN else "WARNING: no HF token set -- gated downloads below will fail.")

## 3. Project path

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "IMTalker").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "IMTalker").exists(), (
    f"Could not find IMTalker/ from {PROJECT_ROOT} -- run from inside the project, "
    "or set PROJECT_ROOT by hand."
)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)

## 4. Download PersonaPlex (4-bit weights + bundled `moshi` source)

Mirrors exactly the parts of `prepare_imtalker_personaplex.sh` that this
notebook actually needs (its steps [4/9] and [5/9], plus its final
`pip install -e .../moshi` step) -- skipping the renderer/generator/voice
checkpoints that script also downloads for the live avatar server, which
this training notebook has no use for.

In [ ]:
from ref_lora_training.common.proc_utils import run_streaming

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
MOSHI_ROOT = CHECKPOINT_DIR / "personaplex_bnb4"
MOSHI_ROOT.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["HF_HOME"] = str(PROJECT_ROOT / ".cache" / "huggingface")
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

steps = [
    (["hf", "download", "brianmatzelle/personaplex-7b-v1-bnb-4bit",
      "--local-dir", str(MOSHI_ROOT)], "PersonaPlex bnb4 weights + bundled moshi source"),
    (["hf", "download", "nvidia/personaplex-7b-v1",
      "tokenizer-e351c8d8-checkpoint125.safetensors", "tokenizer_spm_32k_3.model",
      "--local-dir", str(MOSHI_ROOT)], "PersonaPlex Mimi codec + text tokenizer"),
]
for cmd, description in steps:
    print(f"--- {description} ---", flush=True)
    code_ = run_streaming(cmd, env=env, prefix="[download]")
    if code_ != 0:
        raise RuntimeError(f"download step failed ({description}): exit code {code_}")

moshi_src = MOSHI_ROOT / "moshi"
assert (moshi_src / "pyproject.toml").exists(), (
    f"expected bundled moshi source at {moshi_src}, but pyproject.toml is missing -- "
    "the brianmatzelle/personaplex-7b-v1-bnb-4bit download may be incomplete."
)
print("--- installing bundled moshi (editable, --no-deps) ---", flush=True)
code_ = run_streaming(
    [sys.executable, "-m", "pip", "install", "-e", str(moshi_src), "--no-deps"], prefix="[pip]",
)
if code_ != 0:
    raise RuntimeError(f"pip install -e {moshi_src} failed: exit code {code_}")
print("PersonaPlex download + moshi install complete.")

## 5. Configuration

In [ ]:
MIMI_HF_REPO = "nvidia/personaplex-7b-v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUANTIZE_4BIT = True
NUM_CODEBOOKS = 8          # fallback only -- the real count is read directly off the
                           # loaded model (lm.num_codebooks, confirmed via source inspection;
                           # see model_adapter.resolve_num_codebooks). This just seeds the
                           # tiny synthetic contract-check batch before that model is loaded.

# --- Dataset ------------------------------------------------------------
DATASET_DIR = PROJECT_ROOT / "ref_lora_training" / "dataset_out"
TRAIN_PATH = DATASET_DIR / "train.jsonl"
VAL_PATH = DATASET_DIR / "val.jsonl"
MAX_SEQ_LEN = 512          # tokens (== frames, see batching.py); truncates from the front

# --- LoRA -----------------------------------------------------------------
LORA_RANK = 16
LORA_ALPHA = None          # None -> 2 * rank, matching IMTalker/generator/train_lora.py's convention
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["proj", "fc1", "out_proj", "fc2", "linear", "in_proj"]
# ^ the exact target_modules list from the currently-deployed reference LoRA's own
# adapter_config.json (see prepare_imtalker_personaplex.sh) -- confirmed-correct
# module names for this exact architecture. Set to None to fall back to
# auto-discovery instead (common/model_adapter.discover_target_modules).

# --- Training ---------------------------------------------------------------
OUTPUT_REF_LORA_DIR = PROJECT_ROOT / "ref_lora_training" / "checkpoints_out" / "rag_lora"
BATCH_SIZE = 4             # per GPU
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
LOG_EVERY = 10
EVAL_EVERY = 100
SAVE_EVERY = 200

# None -> auto-detect and use every visible GPU. Set an int to cap it.
N_GPUS = None

print("Config loaded. DEVICE =", DEVICE, " visible GPUs =", torch.cuda.device_count())

## 6. Contract check -- DO NOT SKIP

Quick, cheap, single-GPU sanity check run directly in this notebook (not
through the multi-GPU launcher) so a bad assumption fails fast, before
spending real GPU-hours: loads the base model once, attaches a LoRA, and runs
one real forward+backward pass on a tiny synthetic batch. If this cell
raises, read the error message and `ref_lora_training/common/model_adapter.py`'s
module docstring before doing anything else.

The full training run in Section 8 repeats this same check independently on
every GPU it uses (see `common/train_worker.py`) -- this cell is just a fast
single-GPU preview so you don't have to wait for the multi-GPU launch to find
out.

In [ ]:
from ref_lora_training.common.model_adapter import load_base_model, attach_lora, run_contract_check, resolve_vocab_size

_base = load_base_model(
    moshi_root=str(MOSHI_ROOT), mimi_hf_repo=MIMI_HF_REPO, device=DEVICE,
    quantize_4bit=QUANTIZE_4BIT, num_codebooks=NUM_CODEBOOKS, load_mimi=False,
)
_lm, tokenizer = _base.lm, _base.tokenizer
print("model_type:", _base.model_type)
print("base params (B):", sum(p.numel() for p in _lm.parameters()) / 1e9)

_peft_model = attach_lora(
    _lm, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES,
)
_vocab_size_guess = resolve_vocab_size(tokenizer)
_forward_attempt = run_contract_check(_peft_model, _vocab_size_guess, device=DEVICE, num_codebooks=NUM_CODEBOOKS)
print("Contract check passed. Using forward convention:", _forward_attempt)

In [ ]:
# Free this notebook process's copy before the multi-GPU launch below spawns
# its own fresh worker processes -- otherwise this GPU would be holding two
# copies of the model at once (this one plus whichever torchrun worker lands
# on the same physical GPU).
import gc

del _peft_model, _lm
gc.collect()
torch.cuda.empty_cache()
print("Freed the contract-check model's GPU memory.")

## 7. Load the dataset (sanity check only -- the real loading happens per-GPU in Section 8)

In [ ]:
from ref_lora_training.common.dataset_builder import read_jsonl
from collections import Counter

train_episodes = read_jsonl(TRAIN_PATH)
val_episodes = read_jsonl(VAL_PATH) if VAL_PATH.exists() else []
assert train_episodes, f"no training episodes found at {TRAIN_PATH} -- run 01_Dataset_Generation.ipynb first"
print(f"train episodes: {len(train_episodes)}   val episodes: {len(val_episodes)}")
print("train composition:", dict(Counter(e.example_type for e in train_episodes)))

## 8. Launch multi-GPU training

Auto-detects every visible GPU (or uses `N_GPUS` if you set it above) and
launches `common/train_worker.py` under `torchrun`, one process per GPU, each
with its own full 4-bit model copy and LoRA, gradients averaged across GPUs
every optimizer step (`DistributedDataParallel`). Watch `nvidia-smi` in
another terminal while this runs -- every GPU should be near 100% utilization.

Output from every rank streams live below; only rank 0 prints the periodic
loss/eval lines (`[train_worker] ...`) to keep the log readable, but every
rank's own startup/contract-check/error output is included too.

In [ ]:
from ref_lora_training.common.launch_training import launch_ddp_training

launch_ddp_training(
    moshi_root=str(MOSHI_ROOT),
    mimi_hf_repo=MIMI_HF_REPO,
    quantize_4bit=QUANTIZE_4BIT,
    num_codebooks=NUM_CODEBOOKS,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODULES,
    train_path=str(TRAIN_PATH),
    val_path=str(VAL_PATH) if VAL_PATH.exists() else "",
    max_seq_len=MAX_SEQ_LEN,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    log_every=LOG_EVERY,
    eval_every=EVAL_EVERY,
    save_every=SAVE_EVERY,
    output_ref_lora_dir=str(OUTPUT_REF_LORA_DIR),
    n_gpus=N_GPUS,
    hf_token=HF_TOKEN,
)

## 9. Quick qualitative check

Reloads the base model fresh (single GPU -- the training subprocesses from
Section 8 have exited and freed their GPUs by now) plus the adapter that was
just trained, and feeds a couple of hand-written `<ref>` blocks through it in
plain teacher-forcing mode (not the full streaming avatar pipeline) as a fast
sanity check. This checks the model's next-token predictions after a `<ref>`
block look like real words and not garbage -- it is NOT a substitute for
testing with `ENABLE_SEARCH=1` against the real pipeline and the scenarios in
`logs/detailed_20260908_054904.log` before calling this done.

In [ ]:
from peft import PeftModel
from ref_lora_training.common.ref_format import wrap_with_ref_tags

_qc_base = load_base_model(
    moshi_root=str(MOSHI_ROOT), mimi_hf_repo=MIMI_HF_REPO, device=DEVICE,
    quantize_4bit=QUANTIZE_4BIT, num_codebooks=NUM_CODEBOOKS, load_mimi=False,
)
qc_tokenizer = _qc_base.tokenizer
qc_model = PeftModel.from_pretrained(_qc_base.lm, str(OUTPUT_REF_LORA_DIR / "lora"))
qc_model.eval()
print("Loaded the freshly trained adapter for a quick check.")

def quick_check(user_text: str, ref_fact: str):
    prompt_text = user_text.strip() + "\n" + wrap_with_ref_tags(ref_fact)
    ids = qc_tokenizer.encode(prompt_text)
    print("PROMPT:", prompt_text)
    print("-> (verify this reads like the start of a real, grounded spoken answer,")
    print("    with no literal <ref>/<lookup> text and no leftover markup)")
    # NOTE: this only exercises the text-forward path validated by the contract
    # check above; it does not run the full streaming avatar generation loop.
    # For a true end-to-end check, load this adapter into liveTry.py itself
    # (REF_LORA_DIR=<OUTPUT_REF_LORA_DIR's parent>) and replay the questions
    # from logs/detailed_20260908_054904.log with ENABLE_SEARCH=1.

quick_check("What is Bitcoin trading at right now?", "Bitcoin is trading at $71,204 today.")
quick_check("How is Tesla stock doing today?", "Tesla stock is trading at $309.32 today.")
print("\nReminder: the authoritative test is replaying logs/detailed_20260908_054904.log's")
print("turns against the live server with this adapter loaded.")

## 10. Deploy

Copy (or symlink) the trained adapter to where `run_imtalker_personaplex.sh`
expects it:

```bash
REF_LORA_DIR=/path/to/deployed/rag_lora
mkdir -p "$REF_LORA_DIR"
cp -r ref_lora_training/checkpoints_out/rag_lora/lora "$REF_LORA_DIR/lora"
```

Then launch with `ENABLE_SEARCH=1 REF_LORA_DIR=$REF_LORA_DIR WEB_SEARCH_API_KEY=... ./run_imtalker_personaplex.sh`
and replay the turns from `logs/detailed_20260908_054904.log` by voice, checking
specifically for the three original failure modes: silence after injection,
tag/markup leaking into speech, and a `<ref>` fact bleeding into unrelated
later turns.